# Figure: gradients flow across tree rebuilds

Loads `results/differentiability/nn_rebuild.{json,npz}` (`bench/differentiability/nn_rebuild.py`). Never recomputes.

Gradient descent on a nearest-neighbour spacing objective, with the tree rebuilt inside the differentiated function at every step. Descent stays smooth across the many steps on which the discrete tree ordering changes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_here = Path.cwd()
_candidates = [
    _here / "results" / "differentiability",
    _here.parents[1] / "results" / "differentiability",
]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})
PALETTE = {"radix": "#4C72B0", "octree": "#DD8452", "kdtree": "#55A868"}
BASELINE = "#8C8C8C"

In [ ]:
data = json.loads((RESULTS / "nn_rebuild.json").read_text())
meta = data["metadata"]; params = data["params"]; summary = data["summary"]; traj = data["trajectory"]

arrays = np.load(RESULTS / "nn_rebuild.npz")
pos_initial = arrays["positions_initial"]; pos_final = arrays["positions_final"]

steps = np.asarray(traj["step"])
loss = np.asarray(traj["loss"])
mean_nn = np.asarray(traj["mean_nn"])
changed = np.asarray(traj["topology_changed"], dtype=bool)
target = params["target_distance"]
print(f"device={meta['device_kind']} jax={meta['jax_version']} commit={meta.get('git_commit')}")
print(f"N={params['num_particles']} steps={params['steps']} topology_changes={summary['topology_changes']}")

In [ ]:
disp = pos_final - pos_initial
n_sample = min(64, pos_initial.shape[0])
sample = np.arange(n_sample)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)

# (a) initial vs final positions
axes[0].scatter(pos_initial[:, 0], pos_initial[:, 1], s=14, alpha=0.6, color=BASELINE, label="initial")
axes[0].scatter(pos_final[:, 0], pos_final[:, 1], s=14, alpha=0.75, color=PALETTE["radix"], label="final")
axes[0].set_title("(a) initial vs. final positions")
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[0].set_aspect("equal", adjustable="box"); axes[0].legend(loc="best")

# (b) displacement vectors
axes[1].scatter(pos_initial[:, 0], pos_initial[:, 1], s=8, alpha=0.15, color=BASELINE)
axes[1].quiver(
    pos_initial[sample, 0], pos_initial[sample, 1], disp[sample, 0], disp[sample, 1],
    angles="xy", scale_units="xy", scale=1.0, width=0.004, color=PALETTE["radix"], alpha=0.85,
)
axes[1].set_title(f"(b) displacements (sample of {n_sample})")
axes[1].set_xlabel("$x$"); axes[1].set_ylabel("$y$")
axes[1].set_aspect("equal", adjustable="box")

# (c) trajectory with topology-change markers
ax = axes[2]
ax.plot(steps, loss, color=PALETTE["radix"], label="loss")
ax.set_yscale("log")
ax.set_xlabel("optimizer step")
ax.set_ylabel("loss (log scale)", color=PALETTE["radix"])
ax.tick_params(axis="y", labelcolor=PALETTE["radix"])

ax2 = ax.twinx()
ax2.plot(steps, mean_nn, color=PALETTE["octree"], label="mean NN distance")
ax2.axhline(target, color=BASELINE, linestyle="--", linewidth=1.0, label=f"target = {target:g}")
ax2.set_ylabel("mean NN distance", color=PALETTE["octree"])
ax2.tick_params(axis="y", labelcolor=PALETTE["octree"]); ax2.grid(False)

change_steps = steps[changed]
if change_steps.size:
    ax2.scatter(change_steps, mean_nn[changed], s=22, color=PALETTE["kdtree"], zorder=5, label="topology changed")

lines = ax.get_lines() + ax2.get_lines()
labels = [l.get_label() for l in lines] + (["topology changed"] if change_steps.size else [])
handles = lines + (ax2.collections if change_steps.size else [])
ax.legend(handles, labels, loc="upper right")
ax.set_title(f"(c) descent across {summary['topology_changes']} rebuilds (N={params['num_particles']})")

out = FIGDIR / "fig_nn_rebuild.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)